# Data Loading

Load the three datasets we need:

- **PubMedQA** — research-style Q&A, knowledge base A
- **MedQuAD** — consumer-oriented medical Q&A, knowledge base B
- **MedlinePlus** — NLM health topic XML files, additional consumer content

All saved as raw files for preprocessing.

Outputs:
- `data/raw/pubmedqa_raw.csv`
- `data/raw/medquad_raw.csv`
- `data/raw/medlineplus_raw.csv`
- `reports/schema_validation_report.md`

In [1]:
import os
import io
import zipfile
import requests
import pandas as pd
from pathlib import Path
from xml.etree import ElementTree as ET
from datasets import load_dataset

In [2]:
ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
REPORTS_DIR = ROOT / "reports"
RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. PubMedQA

In [3]:
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_artificial")
pubmedqa

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 211269
    })
})

In [4]:
split = list(pubmedqa.keys())[0]
df_pubmed = pubmedqa[split].to_pandas()

print("rows:", len(df_pubmed))
print("cols:", df_pubmed.columns.tolist())

rows: 211269
cols: ['pubid', 'question', 'context', 'long_answer', 'final_decision']


In [5]:
pd.set_option("display.max_colwidth", 120)
df_pubmed.head(2)

,pubid,question,context,long_answer,final_decision
0,25429730,Are group 2 innate lymphoid cells ( ILC2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?,{'contexts': ['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innat...,"As ILC2s are elevated in patients with CRSwNP, they may drive nasal polyp formation in CRS. ILC2s are also linked wi...",yes
1,25433161,Does vagus nerve contribute to the development of steatohepatitis and obesity in phosphatidylethanolamine N-methyltr...,"{'contexts': ['Phosphatidylethanolamine N-methyltransferase (PEMT), a liver enriched enzyme, is responsible for appr...",Neuronal signals via the hepatic vagus nerve contribute to the development of steatohepatitis and protection against...,yes


## 2. MedQuAD

In [6]:
medquad = load_dataset("lavita/MedQuAD")
medquad

DatasetDict({
    train: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 47441
    })
})

In [7]:
split = list(medquad.keys())[0]
df_medquad = medquad[split].to_pandas()

print("rows:", len(df_medquad))
print("cols:", df_medquad.columns.tolist())

rows: 47441
cols: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']


In [8]:
pd.set_option("display.max_colwidth", 120)
df_medquad.head(2)

,document_id,document_source,document_url,category,umls_cui,umls_semantic_types,umls_semantic_group,synonyms,question_id,question_focus,question_type,question,answer
0,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair,None,C0343073,T047,Disorders,KWWH,0000559-1,keratoderma with woolly hair,information,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increa...
1,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair,None,C0343073,T047,Disorders,KWWH,0000559-2,keratoderma with woolly hair,frequency,How many people are affected by keratoderma with woolly hair ?,Keratoderma with woolly hair is rare; its prevalence worldwide is unknown. Type I (Naxos disease) was first describ...


In [9]:
# MedQuAD has ~31k records with empty answers (removed for copyright).
# We'll filter them in preprocessing, but worth seeing the count now.
answer_col = "answer"  # confirm from columns above if different
empty = df_medquad[answer_col].isna() | (df_medquad[answer_col].astype(str).str.strip() == "")

print(f"total:          {len(df_medquad):,}")
print(f"with answer:    {(~empty).sum():,}")
print(f"without answer: {empty.sum():,}")

total:          47,441
with answer:    16,407
without answer: 31,034


## 3. MedlinePlus

NLM publishes health topic XML files daily. We grab the compressed version
and parse it into a flat table. Only English topics for now.

In [10]:
# NLM rotates the filename daily, so we scrape the XML page to find the latest link.
XML_PAGE = "https://medlineplus.gov/xml.html"
html = requests.get(XML_PAGE, timeout=30).text

import re
matches = re.findall(
    r"https://medlineplus\.gov/xml/mplus_topics_compressed_\d{4}-\d{2}-\d{2}\.zip",
    html,
)
zip_url = matches[0]
print("latest:", zip_url)

latest: https://medlineplus.gov/xml/mplus_topics_compressed_2026-10-03.zip


In [11]:
r = requests.get(zip_url, timeout=120)
r.raise_for_status()

z = zipfile.ZipFile(io.BytesIO(r.content))
xml_name = [n for n in z.namelist() if n.endswith(".xml")][0]
xml_bytes = z.read(xml_name)

print("file:", xml_name)
print("size:", len(xml_bytes) / 1024**2, "MB")

file: mplus_topics_2026-10-03.xml
size: 28.749048233032227 MB


In [12]:
root = ET.fromstring(xml_bytes)

rows = []
for topic in root.findall("health-topic"):
    if topic.get("language") != "English":
        continue

    summary_el = topic.find("full-summary")
    summary = "".join(summary_el.itertext()).strip() if summary_el is not None else ""

    groups = " | ".join(
        g.text.strip() for g in topic.findall("group") if g.text
    )

    also_called = " | ".join(
        s.text.strip() for s in topic.findall("also-called") if s.text
    )

    rows.append({
        "title": topic.get("title", ""),
        "url": topic.get("url", ""),
        "meta_desc": topic.get("meta-desc", ""),
        "summary": summary,
        "groups": groups,
        "also_called": also_called,
    })

df_medline = pd.DataFrame(rows)
print("rows:", len(df_medline))

rows: 1017


In [13]:
pd.set_option("display.max_colwidth", 200)
df_medline.head(3)

,title,url,meta_desc,summary,groups,also_called
0,A1C,https://medlineplus.gov/a1c.html,"A1C is a blood test for type 2 diabetes and prediabetes. It measures your average blood glucose, or blood sugar, levels over the past 3 months. Learn more.","<p>A1C is a blood test for <a href=""https://medlineplus.gov/diabetestype2.html"">type 2 diabetes</a> and <a href=""https://medlineplus.gov/prediabetes.html"">prediabetes</a>. It measures your average...",Diagnostic Tests | Diabetes Mellitus,Glycohemoglobin | HbA1C | Hemoglobin A1C test
1,Abdominal Pain,https://medlineplus.gov/abdominalpain.html,Stomach aches can be painful. Find out what might be the cause of your abdominal pain.,"<p>Your abdomen extends from below your chest to your groin. Some people call it the stomach, but your abdomen contains many other important organs. Pain in the abdomen can come from any one of th...",Digestive System | Symptoms,Bellyache
2,Abortion,https://medlineplus.gov/abortion.html,An abortion is a medical procedure to end a pregnancy. It uses medicine or surgery to remove the embryo or fetus and placenta from the uterus.,"<p>An induced abortion is a procedure to end a pregnancy. It can be done two different ways:</p>\n<ul>\n<li><a href=""https://medlineplus.gov/ency/article/007382.htm"">Medication abortion</a> (also ...",Pregnancy and Reproduction | Female Reproductive System,Induced Abortion


## 4. Save raw copies

In [14]:
pubmed_path = RAW_DIR / "pubmedqa_raw.csv"
medquad_path = RAW_DIR / "medquad_raw.csv"
medline_path = RAW_DIR / "medlineplus_raw.csv"

df_pubmed.to_csv(pubmed_path, index=False)
df_medquad.to_csv(medquad_path, index=False)
df_medline.to_csv(medline_path, index=False)

for p in [pubmed_path, medquad_path, medline_path]:
    print(f"{p.name}: {os.path.getsize(p) / 1024**2:.1f} MB")

pubmedqa_raw.csv: 452.0 MB
medquad_raw.csv: 31.7 MB
medlineplus_raw.csv: 2.9 MB


## 5. Report

In [15]:
report = f"""# Schema Validation Report

## PubMedQA (`qiaojin/PubMedQA`, `pqa_artificial`)
- Rows: {len(df_pubmed):,}
- Columns: {df_pubmed.columns.tolist()}
- Nulls total: {df_pubmed.isnull().sum().sum():,}

## MedQuAD (`lavita/MedQuAD`)
- Rows: {len(df_medquad):,}
- Columns: {df_medquad.columns.tolist()}
- Records with answers: {(~empty).sum():,}
- Records without answer: {empty.sum():,}

## MedlinePlus (NLM XML)
- Source: {zip_url}
- Rows (English only): {len(df_medline):,}
- Columns: {df_medline.columns.tolist()}
"""

(REPORTS_DIR / "schema_validation_report.md").write_text(report)
print("report saved")

report saved


Next: `02_preprocessing.ipynb`